# Phase 1 — SDSS DR17 SQL Pipeline & Scaling Relations Atlas

This notebook runs the Phase 1 pipeline end-to-end:
1. Inspect the configured FlatLambdaCDM cosmology ($H_0 = 70\,\mathrm{km\,s^{-1}\,Mpc^{-1}}$, $\Omega_{m0} = 0.3$) and SDSS DR17 volume-limited selection cuts.
2. Query SDSS DR17 via `astroquery.sdss`, clean sentinel values, and compute physical parameters (`re_kpc` from `petroR50_r`, `concentration = petroR90_r / petroR50_r`, `abs_mag_r`, colors, and `morph_proxy`).
3. Populate the normalized SQLite database (`data/galaxy_atlas.db`).
4. Execute analytical SQL queries into Pandas and visualize galaxy scaling relations.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import pandas as pd

from src.acquisition import SDSSCatalogFetcher
from src.analysis import (
    load_galaxy_zoo_agreement,
    load_morphology_summary,
    load_redshift_binned_stats,
    load_scaling_relations_sample,
    plot_color_magnitude,
    plot_concentration_morphology,
    plot_mass_size_relation,
)
from src.config import ProjectConfig
from src.database import GalaxyDatabase

config = ProjectConfig()
fetcher = SDSSCatalogFetcher(config)
print("Database path:", config.db_path)
print("Cosmology:", config.cosmology)
print("Selection cuts:", config.quality_cuts_metadata())

## 1. Inspect the SDSS DR17 SQL Query

In [ ]:
TOP_N = 100  # Set to 2500 for the full Phase 1 catalog
print(fetcher.build_sql_query(top_n=TOP_N))

## 2. Fetch SDSS DR17 Catalog & Populate `data/galaxy_atlas.db`

In [ ]:
result = fetcher.run_pipeline(top_n=TOP_N)
print(f"Raw rows: {result['raw_count']} | Clean rows: {result['clean_count']} | Upserted: {result['upserted_count']}")
result["summary"]

## 3. Analytical SQL Queries into Pandas

In [ ]:
with GalaxyDatabase(config.db_path) as db:
    print("Table counts:", db.get_table_counts())
    summary_df = load_morphology_summary(db)
    sample_df = load_scaling_relations_sample(db)
    z_bins_df = load_redshift_binned_stats(db, bin_width=0.02)
    zoo_df = load_galaxy_zoo_agreement(db)

display(summary_df)
display(z_bins_df)
display(zoo_df)

## 4. Diagnostic Scaling-Relation Figures

In [ ]:
config.ensure_directories()
plot_color_magnitude(sample_df, config.figures_dir / "color_magnitude_diagram.png")
plot_mass_size_relation(sample_df, config.figures_dir / "mass_size_relation.png")
plot_concentration_morphology(sample_df, config.figures_dir / "concentration_morphology.png")
plt.show()